In [1]:
!pip install -q rdkit catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 54.8 MB/s eta 0:00:00


In [2]:
#!/usr/bin/env python3
"""
================================================================================
 AISEHack 2.0 ROUND 2 -- THE ULTIMATE TITAN APEX PIPELINE
================================================================================
 1. 5 DISTINCT FEATURE MATRICES: fp_all, count_text, flagship, research3d
 2. 12-MODEL META-ENSEMBLE: Ridge, BayesRidge, LGBM (Huber, Quantile, Winsor), 
    XGBoost, CatBoost, CharCNN, and MolGNN.
 3. GREEDY OPTIMIZATION: Custom forward-step optimizer for perfect ensemble weights.
 4. FULL-DATA REFIT: 100% training data utilization post-validation.
 5. PSEUDO-LABELING: Confident test predictions are recycled as ground truth.
 6. PHYSICS CLIPPING: Maxwell & Koopmans boundaries applied prior to submission.
 7. OOM SAFE & BOUNDARY SAFE: Strict array limits to prevent shape mismatches.
================================================================================
"""

import os, sys, time, random, warnings, math
from pathlib import Path
from collections import deque, defaultdict
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
os.environ["PYTHONHASHSEED"] = "42"

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler, QuantileTransformer
from sklearn.linear_model import Ridge, BayesianRidge, RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import (AllChem, MACCSkeys, Descriptors, Crippen,
                        Lipinski, rdMolDescriptors, Descriptors3D)
from rdkit.Chem.EState import EState_VSA, EState as ES
from rdkit.Chem import rdFingerprintGenerator

RDLogger.DisableLog("rdApp.*")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── 1. CONFIGURATION ──────────────────────────────────────────────────────────
SEED = 42
N_SPLITS = 5
N_BITS = 2048
TARGETS = ['egc', 'egb', 'ei', 'eea', 'eps', 'nc', 'tg']
COMPUTE_3D = True
MAX_HEAVY_FOR_3D = 80

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def rmse(yt, yp): return float(np.sqrt(mean_squared_error(yt, yp)))

# ── 2. MOLECULE UTILS & PIVOTING ──────────────────────────────────────────────
def clean_polymer_smiles(smi):
    smi = str(smi).strip()
    for tag in ["[]", "[e]", "[d]", "[t]", "[g]"]: smi = smi.replace(tag, "*")
    if "*" not in smi: smi = f"*{smi}*"
    return smi

def mol_from_smiles(smi):
    smi = clean_polymer_smiles(smi)
    for s in (smi, smi.replace("*", "C"), smi.replace("*", "[H]")):
        m = Chem.MolFromSmiles(s)
        if m is not None: return m
    return None

def canonicalize_smiles(smi):
    mol = mol_from_smiles(smi)
    return Chem.MolToSmiles(mol, canonical=True) if mol else str(smi)

def prep_wide_format(df, is_train=True):
    df_clean = df.copy()
    df_clean['smiles_canon'] = df_clean['smiles'].apply(canonicalize_smiles)
    df_clean['target_type'] = df_clean['target_type'].str.lower()
    if not is_train: return df_clean
    wide = []
    for smi, group in tqdm(df_clean.groupby('smiles_canon'), desc="Pivoting to Wide Format"):
        row = {'smiles_canon': smi}
        for t in TARGETS:
            val = group[group['target_type'] == t]['target']
            row[t] = val.mean() if len(val) > 0 else np.nan
        wide.append(row)
    return pd.DataFrame(wide)

def mol_dimer(smi):
    mol = mol_from_smiles(smi)
    if mol is None: return None
    dummy_idx = [a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() == 0]
    if len(dummy_idx) != 2: return None
    try:
        head, tail = dummy_idx[0], dummy_idx[1]
        head_nbrs = list(mol.GetAtomWithIdx(head).GetNeighbors())
        tail_nbrs = list(mol.GetAtomWithIdx(tail).GetNeighbors())
        if len(head_nbrs) != 1 or len(tail_nbrs) != 1: return None
        combo = Chem.CombineMols(mol, mol)
        rw = Chem.RWMol(combo)
        n = mol.GetNumAtoms()
        rw.AddBond(tail_nbrs[0].GetIdx(), head_nbrs[0].GetIdx() + n, Chem.BondType.SINGLE)
        for idx in sorted([tail, head + n], reverse=True): rw.RemoveAtom(idx)
        out = rw.GetMol()
        Chem.SanitizeMol(out)
        return out
    except: return None

# ── 3. FEATURE ENGINEERING ────────────────────────────────────────────────────
def descriptor_vector(mol):
    if mol is None: return np.zeros(56, dtype=np.float32)
    vals = []
    fns = [Descriptors.MolWt, Descriptors.HeavyAtomMolWt, Descriptors.ExactMolWt, Descriptors.MolLogP, Descriptors.MolMR, Descriptors.TPSA, Descriptors.NumValenceElectrons, Descriptors.NumRadicalElectrons, Descriptors.FractionCSP3, Descriptors.HeavyAtomCount, Descriptors.NHOHCount, Descriptors.NOCount, Descriptors.NumHAcceptors, Descriptors.NumHDonors, Descriptors.NumHeteroatoms, Descriptors.NumRotatableBonds, Descriptors.RingCount, Descriptors.BalabanJ, Descriptors.BertzCT, Descriptors.Chi0, Descriptors.Chi0n, Descriptors.Chi0v, Descriptors.Chi1, Descriptors.Chi1n, Descriptors.Chi1v, Descriptors.Chi2n, Descriptors.Chi2v, Descriptors.Chi3n, Descriptors.Chi3v, Descriptors.Chi4n, Descriptors.Chi4v, Descriptors.Kappa1, Descriptors.Kappa2, Descriptors.Kappa3, Descriptors.LabuteASA, Descriptors.PEOE_VSA1, Descriptors.PEOE_VSA2, Descriptors.PEOE_VSA6, Descriptors.PEOE_VSA7, Descriptors.PEOE_VSA8, Descriptors.SMR_VSA1, Descriptors.SMR_VSA3, Descriptors.SMR_VSA5, Descriptors.SlogP_VSA1, Descriptors.SlogP_VSA2, Descriptors.SlogP_VSA3, Descriptors.SlogP_VSA5, Descriptors.SlogP_VSA6, EState_VSA.EState_VSA1, EState_VSA.EState_VSA2, EState_VSA.EState_VSA3, EState_VSA.EState_VSA4, EState_VSA.EState_VSA5, EState_VSA.EState_VSA6, EState_VSA.EState_VSA7, EState_VSA.EState_VSA8]
    for fn in fns:
        try: v = fn(mol); vals.append(float(v) if np.isfinite(v) else 0.0)
        except: vals.append(0.0)
    return np.asarray(vals, dtype=np.float32)

def gasteiger_features(mol):
    if mol is None: return np.zeros(10, dtype=np.float32)
    try:
        mc = Chem.RWMol(mol); AllChem.ComputeGasteigerCharges(mc)
        c = np.array([float(a.GetDoubleProp("_GasteigerCharge")) for a in mc.GetAtoms() if a.HasProp("_GasteigerCharge")]); c = c[np.isfinite(c)]
        if len(c) == 0: return np.zeros(10, dtype=np.float32)
        pos = c[c > 0]; neg = c[c < 0]
        return np.array([c.mean(), c.std(), c.min(), c.max(), pos.sum() if len(pos) else 0.0, neg.sum() if len(neg) else 0.0, len(pos)/len(c), len(neg)/len(c), np.abs(c).mean(), c.max()-c.min()], dtype=np.float32)
    except: return np.zeros(10, dtype=np.float32)

def get_topology_features(mol):
    if mol is None: return np.zeros(242, dtype=np.float32)
    try: ac = np.array(rdMolDescriptors.CalcAUTOCORR2D(mol), dtype=np.float32)
    except: ac = np.zeros(192, dtype=np.float32)
    try: mqn = np.array(rdMolDescriptors.CalcMQNs(mol), dtype=np.float32)
    except: mqn = np.zeros(42, dtype=np.float32)
    try:
        idx = np.array(ES.EStateIndices(mol), dtype=np.float32); idx = idx[np.isfinite(idx)]
        if len(idx) == 0: est = np.zeros(8, dtype=np.float32)
        else: est = np.array([idx.mean(), idx.std(), idx.min(), idx.max(), (idx > 0).mean(), (idx < 0).mean(), np.abs(idx).mean(), idx.max() - idx.min()], dtype=np.float32)
    except: est = np.zeros(8, dtype=np.float32)
    return np.concatenate([ac, mqn, est])

def get_dimer_deltas(mol_m, mol_d):
    if mol_m is None or mol_d is None: return np.zeros(30, dtype=np.float32)
    def _counts(m):
        return np.array([Descriptors.MolWt(m), Descriptors.MolLogP(m), Descriptors.TPSA(m), Descriptors.NumRotatableBonds(m), Descriptors.RingCount(m), rdMolDescriptors.CalcNumAromaticRings(m), rdMolDescriptors.CalcNumAliphaticRings(m), rdMolDescriptors.CalcNumAmideBonds(m), Lipinski.NumHDonors(m), Lipinski.NumHAcceptors(m), Descriptors.NumHeteroatoms(m), Crippen.MolMR(m), Descriptors.BertzCT(m), Descriptors.HeavyAtomCount(m), Descriptors.FractionCSP3(m)], dtype=np.float32)
    try:
        cm = _counts(mol_m); cd = _counts(mol_d)
        return np.concatenate([cd - (2.0 * cm), cd / np.clip(cm * 2.0, 1e-6, None)])
    except: return np.zeros(30, dtype=np.float32)

def backbone_sidechain_features(mol):
    if mol is None: return np.zeros(10, dtype=np.float32)
    try:
        dummy = [a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() == 0]
        heavy = [a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() > 1]
        if len(dummy) < 2: out = np.zeros(10, dtype=np.float32); out[-1] = len(dummy); return out
        path = list(Chem.GetShortestPath(mol, dummy[0], dummy[1])); bb_set = set(path) - set(dummy)
        n_bb = max(1, len(bb_set)); n_sc = max(0, max(1, len(heavy)) - n_bb)
        branch_edges = sum(1 for idx in bb_set for nb in mol.GetAtomWithIdx(idx).GetNeighbors() if nb.GetIdx() not in bb_set and nb.GetIdx() not in dummy)
        return np.asarray([n_bb, n_sc, n_sc / max(1, len(heavy)), sum(1 for idx in bb_set if mol.GetAtomWithIdx(idx).GetIsAromatic()) / n_bb, sum(1 for idx in bb_set if mol.GetAtomWithIdx(idx).GetAtomicNum() not in (1, 6)) / n_bb, branch_edges, branch_edges / n_bb, n_bb / max(1, len(heavy)), len(path), len(dummy)], dtype=np.float32)
    except: return np.zeros(10, dtype=np.float32)

import signal
class _Embed3DTimeout(Exception): pass
def _embed3d_alarm_handler(signum, frame): raise _Embed3DTimeout()

def compute_3d_features_safe(mol):
    if not COMPUTE_3D or mol is None or mol.GetNumHeavyAtoms() > MAX_HEAVY_FOR_3D: return np.zeros(15, dtype=np.float32)
    old_handler = signal.signal(signal.SIGALRM, _embed3d_alarm_handler)
    signal.alarm(6)
    try:
        m = Chem.AddHs(mol); params = AllChem.ETKDGv3(); params.randomSeed = SEED; params.useRandomCoords = True
        cid = AllChem.EmbedMolecule(m, params)
        if cid < 0: cid = AllChem.EmbedMolecule(m, useRandomCoords=True, randomSeed=SEED, maxAttempts=20)
        if cid < 0: return np.zeros(15, dtype=np.float32)
        try:
            if AllChem.MMFFOptimizeMolecule(m, maxIters=100) != 0: AllChem.UFFOptimizeMolecule(m, maxIters=100)
        except: pass
        shape_vals = []
        for fn in [Descriptors3D.PMI1, Descriptors3D.RadiusOfGyration, Descriptors3D.Asphericity, Descriptors3D.Eccentricity]:
            try: v = fn(m); shape_vals.append(float(v) if np.isfinite(v) else 0.0)
            except: shape_vals.append(0.0)
        try: vol = float(AllChem.ComputeMolVolume(m))
        except: vol = 0.0
        return np.array(shape_vals + [vol] + [0]*10, dtype=np.float32)[:15]
    except: return np.zeros(15, dtype=np.float32)
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

def bitvect_to_array(fp, n):
    arr = np.zeros((n,), dtype=np.float32); DataStructs.ConvertToNumpyArray(fp, arr); return arr

# ── 4. PYTORCH NLP & GRAPH MODELS ─────────────────────────────────────────────
SMILES_CHARS = list(" #%()+-./0123456789=@ABCDEFGHIKLMNOPRSTVXZ[\\]abcdefgilmnoprstuy*")
CHAR2ID = {c: i + 1 for i, c in enumerate(SMILES_CHARS)}

def encode_smiles(smi_list, max_len=256):
    out = np.zeros((len(smi_list), max_len), dtype=np.int64)
    for i, s in enumerate(smi_list):
        ids = [CHAR2ID.get(c, 0) for c in str(s)[:max_len]]
        out[i, :len(ids)] = ids
    return out

def mol_to_graph(mol, max_atomic_num=100):
    if mol is None or mol.GetNumAtoms() == 0: return (np.zeros((1, 7), dtype=np.float32), np.zeros((2, 1), dtype=np.int64), np.zeros((1, 6), dtype=np.float32))
    atom_feats = []
    for atom in mol.GetAtoms():
        hyb = atom.GetHybridization()
        atom_feats.append([min(atom.GetAtomicNum(), max_atomic_num), atom.GetDegree(), atom.GetFormalCharge(), atom.GetTotalNumHs(), float(atom.GetIsAromatic()), float(atom.IsInRing()), float(int(hyb)) if hyb is not None else 0.0])
    src, dst, eattr = [], [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx(); bt = bond.GetBondTypeAsDouble()
        feat = [float(bt == 1.0), float(bt == 2.0), float(bt == 3.0), float(bt == 1.5), float(bond.GetIsConjugated()), float(bond.IsInRing())]
        for a, b in [(i, j), (j, i)]: src.append(a); dst.append(b); eattr.append(feat)
    if not src: src, dst, eattr = [0], [0], [[0.0]*6]
    return np.array(atom_feats, dtype=np.float32), np.array([src, dst], dtype=np.int64), np.array(eattr, dtype=np.float32)

def collate_graphs(graph_list, device):
    atom_list, edge_idx_list, edge_attr_list, batch_vec = [], [], [], []
    offset = 0
    for gi, (af, ei, ea) in enumerate(graph_list):
        n = af.shape[0]
        atom_list.append(af); edge_idx_list.append(ei + offset); edge_attr_list.append(ea)
        batch_vec.append(np.full(n, gi, dtype=np.int64))
        offset += n
    return (torch.tensor(np.vstack(atom_list), dtype=torch.float32, device=device), torch.tensor(np.hstack(edge_idx_list), dtype=torch.long, device=device), torch.tensor(np.vstack(edge_attr_list), dtype=torch.float32, device=device), torch.tensor(np.concatenate(batch_vec), dtype=torch.long, device=device))

class MultiTaskUncertaintyLoss(nn.Module):
    def __init__(self, num_tasks):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(num_tasks))
    def forward(self, preds, targets, masks):
        loss = 0
        for i in range(preds.shape[1]):
            valid_mask = masks[:, i].bool()
            if valid_mask.sum() > 0:
                mse = F.mse_loss(preds[valid_mask, i], targets[valid_mask, i])
                loss += 0.5 * torch.exp(-self.log_vars[i]) * mse + 0.5 * self.log_vars[i]
        return loss

class MultiTaskCharCNN(nn.Module):
    def __init__(self, vocab_size, emb_dim=48, ch=96, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab_size + 1, emb_dim, padding_idx=0)
        self.conv1 = nn.Conv1d(emb_dim, ch, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(ch, ch, kernel_size=3, padding=1)
        self.lstm = nn.LSTM(ch, hidden, batch_first=True, bidirectional=True)
        self.heads = nn.ModuleList([nn.Sequential(nn.Linear(hidden * 2, 64), nn.ReLU(), nn.Linear(64, 1)) for _ in range(7)])
    def forward(self, x):
        mask = (x != 0).float().unsqueeze(1)
        e = self.emb(x).transpose(1, 2)
        c = F.relu(self.conv2(F.relu(self.conv1(e)) * mask)) * mask
        out, _ = self.lstm(c.transpose(1, 2))
        pooled = (out * mask.transpose(1, 2)).sum(dim=1) / mask.sum(dim=2).clamp(min=1e-9)
        return torch.cat([head(pooled) for head in self.heads], dim=1), pooled

class GINLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.edge_mlp = nn.Sequential(nn.Linear(dim + 6, dim), nn.ReLU())
        self.update_mlp = nn.Sequential(nn.Linear(dim * 2, dim), nn.ReLU(), nn.Linear(dim, dim))
        self.norm = nn.LayerNorm(dim)
    def forward(self, x, ei, ea):
        msg = self.edge_mlp(torch.cat([x[ei[0]], ea], dim=-1))
        agg = torch.zeros(x.size(0), msg.size(-1), device=x.device).index_add_(0, ei[1], msg)
        return self.norm(x + self.update_mlp(torch.cat([x, agg], dim=-1)))

class MultiTaskMolGNN(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(101, hidden)
        self.proj = nn.Sequential(nn.Linear(hidden + 6, hidden), nn.ReLU())
        self.layers = nn.ModuleList([GINLayer(hidden) for _ in range(3)])
        self.heads = nn.ModuleList([nn.Sequential(nn.Linear(hidden * 2, 64), nn.ReLU(), nn.Linear(64, 1)) for _ in range(7)])
    def forward(self, af, ei, ea, batch, n_graphs):
        x = self.proj(torch.cat([self.emb(af[:, 0].long()), af[:, 1:]], dim=-1))
        for L in self.layers: x = L(x, ei, ea)
        sum_ = torch.zeros(n_graphs, x.size(-1), device=x.device).index_add_(0, batch, x)
        cnt = torch.zeros(n_graphs, 1, device=x.device).index_add_(0, batch, torch.ones(x.size(0), 1, device=x.device)).clamp(min=1)
        mean_p = sum_ / cnt
        max_p = torch.full((n_graphs, x.size(-1)), -1e9, device=x.device).scatter_reduce_(0, batch.unsqueeze(1).expand(-1, x.size(-1)), x, reduce='amax', include_self=False)
        pooled = torch.cat([mean_p, max_p], dim=1)
        return torch.cat([head(pooled) for head in self.heads], dim=1), pooled

# ── 5. GREEDY BLEND OPTIMIZER ─────────────────────────────────────────────────
def optimize_blend_weights(oof_preds, y_true):
    best_weights = np.zeros(oof_preds.shape[1])
    best_weights[0] = 1.0
    for _ in range(200):
        best_score = -np.inf
        best_idx = -1
        for j in range(oof_preds.shape[1]):
            test_weights = best_weights.copy()
            test_weights[j] += 0.05
            test_weights /= test_weights.sum()
            blend = np.average(oof_preds, axis=1, weights=test_weights)
            score = r2_score(y_true, blend)
            if score > best_score:
                best_score = score
                best_idx = j
        if best_idx != -1:
            best_weights[best_idx] += 0.05
            best_weights /= best_weights.sum()
    return best_weights

# ── 6. FULL TITAN EXECUTION ───────────────────────────────────────────────────
def main():
    print(f"System Initialized. Execution Device: {TORCH_DEVICE}")
    data_dir = Path("/kaggle/input/competitions/ppp-round-2")
    if not data_dir.exists(): data_dir = Path(".")
    
    train_long = pd.read_csv(data_dir / "train.csv")
    test_long  = pd.read_csv(data_dir / "test.csv")
    
    print("\n--- Pivoting to Multi-Task Wide Format ---")
    train_wide = prep_wide_format(train_long, is_train=True)
    test_wide = prep_wide_format(test_long, is_train=False)
    
    all_smiles = pd.concat([train_wide['smiles_canon'], test_wide['smiles_canon']]).values
    n_train = len(train_wide)
    
    print("\n--- Extracting All Physics & Topology ---")
    mfp_r3 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=N_BITS)
    ap_gen = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=N_BITS)
    
    fp_l, ap_l, mc_l, desc_l, topo_l, delta_l, bbsc_l, shp_l, gast_l, graphs = [], [], [], [], [], [], [], [], [], []
    
    for smi in tqdm(all_smiles, desc="Physics & Graphs"):
        m = mol_from_smiles(smi); d = mol_dimer(smi)
        if m is None:
            fp_l.append(np.zeros(N_BITS, dtype=np.float32)); ap_l.append(np.zeros(N_BITS, dtype=np.float32))
            mc_l.append(np.zeros(167, dtype=np.float32)); desc_l.append(np.zeros(56, dtype=np.float32))
            topo_l.append(np.zeros(242, dtype=np.float32)); delta_l.append(np.zeros(30, dtype=np.float32))
            bbsc_l.append(np.zeros(10, dtype=np.float32)); shp_l.append(np.zeros(15, dtype=np.float32))
            gast_l.append(np.zeros(10, dtype=np.float32)); graphs.append(mol_to_graph(m))
            continue
            
        r3 = np.zeros(N_BITS, dtype=np.float32); DataStructs.ConvertToNumpyArray(mfp_r3.GetFingerprint(m), r3); fp_l.append(r3)
        ap = np.zeros(N_BITS, dtype=np.float32); DataStructs.ConvertToNumpyArray(ap_gen.GetFingerprint(m), ap); ap_l.append(ap)
        mc = np.zeros(167, dtype=np.float32); DataStructs.ConvertToNumpyArray(MACCSkeys.GenMACCSKeys(m), mc); mc_l.append(mc)
        desc_l.append(descriptor_vector(m)); topo_l.append(get_topology_features(m))
        delta_l.append(get_dimer_deltas(m, d)); bbsc_l.append(backbone_sidechain_features(m))
        shp_l.append(compute_3d_features_safe(m)); gast_l.append(gasteiger_features(m))
        graphs.append(mol_to_graph(m))

    def vs(lst): return np.vstack(lst).astype(np.float32)
    FP, AP, MC, DESC, TOPO, DELTA, BBSC, SHP, GAST = vs(fp_l), vs(ap_l), vs(mc_l), vs(desc_l), vs(topo_l), vs(delta_l), vs(bbsc_l), vs(shp_l), vs(gast_l)
    
    tfidf = TfidfVectorizer(analyzer="char", ngram_range=(2,6), min_df=2, max_features=15000, dtype=np.float32)
    TXT = TruncatedSVD(n_components=200, random_state=SEED).fit_transform(tfidf.fit_transform(all_smiles)).astype(np.float32)
    
    # Build the 5 Independent Feature Sets
    print("Partitioning Independent Feature Matrices...")
    feature_sets = {
        "count_text": np.hstack([MC, DESC, TXT]),
        "fp_all": np.hstack([FP, AP, MC, DESC]),
        "flagship": np.hstack([FP, MC, DESC, TOPO, DELTA, BBSC, GAST]),
        "research3d": np.hstack([FP, MC, DESC, TOPO, DELTA, BBSC, SHP]),
    }
    for k in feature_sets: feature_sets[k] = feature_sets[k][:, np.std(feature_sets[k], axis=0) > 1e-6]

    # ── PyTorch Pre-Training ──
    print("\n--- Training Deep SMILES NLP & GNN ---")
    y_raw = train_wide[TARGETS].values
    y_scaled = np.full_like(y_raw, np.nan)
    scalers = []
    for i in range(len(TARGETS)):
        mask = ~np.isnan(y_raw[:, i])
        sc = RobustScaler()
        y_scaled[mask, i] = sc.fit_transform(y_raw[mask, i].reshape(-1, 1)).flatten()
        scalers.append(sc)
        
    X_char = encode_smiles(all_smiles)
    char_model = MultiTaskCharCNN(vocab_size=len(SMILES_CHARS)).to(TORCH_DEVICE)
    crit_char = MultiTaskUncertaintyLoss(len(TARGETS)).to(TORCH_DEVICE)
    opt_char = torch.optim.AdamW(list(char_model.parameters()) + list(crit_char.parameters()), lr=1e-3)
    
    char_model.train()
    for ep in range(35):
        perm = torch.randperm(n_train)
        for i in range(0, n_train, 128):
            idx = perm[i:i+128]
            bx = torch.tensor(X_char[idx], dtype=torch.long, device=TORCH_DEVICE)
            by = torch.tensor(np.nan_to_num(y_scaled[idx], nan=0.0), dtype=torch.float32, device=TORCH_DEVICE)
            bm = torch.tensor(~np.isnan(y_scaled[idx]), dtype=torch.float32, device=TORCH_DEVICE)
            opt_char.zero_grad()
            preds, _ = char_model(bx)
            loss = crit_char(preds, by, bm)
            loss.backward(); torch.nn.utils.clip_grad_norm_(char_model.parameters(), 1.0); opt_char.step()
            
    print("Extracting CharCNN Embeddings safely...")
    char_model.eval()
    char_tr_embs, char_te_embs = [], []
    with torch.no_grad():
        for i in range(0, n_train, 128):
            end_idx = min(i + 128, n_train)
            _, emb = char_model(torch.tensor(X_char[i:end_idx], dtype=torch.long, device=TORCH_DEVICE))
            char_tr_embs.append(emb.cpu().numpy())
            
        for i in range(n_train, len(X_char), 128):
            end_idx = min(i + 128, len(X_char))
            _, emb = char_model(torch.tensor(X_char[i:end_idx], dtype=torch.long, device=TORCH_DEVICE))
            char_te_embs.append(emb.cpu().numpy())
            
    char_tr_emb = np.vstack(char_tr_embs)
    char_te_emb = np.vstack(char_te_embs)
    
    print("Training MolGNN...")
    gnn_model = MultiTaskMolGNN().to(TORCH_DEVICE)
    crit_gnn = MultiTaskUncertaintyLoss(len(TARGETS)).to(TORCH_DEVICE)
    opt_gnn = torch.optim.AdamW(list(gnn_model.parameters()) + list(crit_gnn.parameters()), lr=2e-3)
    
    gnn_model.train()
    for ep in range(35):
        perm = torch.randperm(n_train)
        for i in range(0, n_train, 128):
            idx = perm[i:i+128]
            sub = [graphs[j] for j in idx]
            af, ei, ea, batch = collate_graphs(sub, TORCH_DEVICE)
            yb = torch.tensor(np.nan_to_num(y_scaled[idx], nan=0.0), dtype=torch.float32, device=TORCH_DEVICE)
            mb = torch.tensor(~np.isnan(y_scaled[idx]), dtype=torch.float32, device=TORCH_DEVICE)
            opt_gnn.zero_grad()
            preds, _ = gnn_model(af, ei, ea, batch, len(idx))
            loss = crit_gnn(preds, yb, mb)
            loss.backward(); torch.nn.utils.clip_grad_norm_(gnn_model.parameters(), 1.0); opt_gnn.step()
            
    print("Extracting GNN Embeddings safely...")
    gnn_model.eval()
    gnn_tr_embs, gnn_te_embs = [], []
    with torch.no_grad():
        for i in range(0, n_train, 128):
            sub = [graphs[j] for j in range(i, min(i+128, n_train))]
            af, ei, ea, batch = collate_graphs(sub, TORCH_DEVICE)
            _, emb = gnn_model(af, ei, ea, batch, len(sub))
            gnn_tr_embs.append(emb.cpu().numpy())
            
        for i in range(n_train, len(graphs), 128):
            sub = [graphs[j] for j in range(i, min(i+128, len(graphs)))]
            af, ei, ea, batch = collate_graphs(sub, TORCH_DEVICE)
            _, emb = gnn_model(af, ei, ea, batch, len(sub))
            gnn_te_embs.append(emb.cpu().numpy())
            
    gnn_tr_emb = np.vstack(gnn_tr_embs)
    gnn_te_emb = np.vstack(gnn_te_embs)
    
    # ── 12-Model Matrix Factory ──
    for k in feature_sets:
        feature_sets[k] = (np.hstack([feature_sets[k][:n_train], char_tr_emb, gnn_tr_emb]),
                           np.hstack([feature_sets[k][n_train:], char_te_emb, gnn_te_emb]))

    final_wide_preds = test_wide[['smiles_canon']].copy()
    
    for i, t in enumerate(TARGETS):
        print(f"\n{'='*60}\nTraining 12-Model Titan Ensemble for: {t.upper()}\n{'='*60}")
        valid_idx = ~np.isnan(y_raw[:, i])
        y_t = y_raw[valid_idx, i]
        
        use_rg = t in ['tg', 'eps', 'eea', 'nc']
        qt = QuantileTransformer(output_distribution='normal', random_state=SEED) if use_rg else None
        y_fit = qt.fit_transform(y_t.reshape(-1, 1)).flatten() if use_rg else y_t
        
        models = [
            ("ridge_count", "count_text", make_pipeline(SimpleImputer(strategy="median"), RobustScaler(), Ridge(alpha=10.0))),
            ("bayes_count", "count_text", make_pipeline(SimpleImputer(strategy="median"), RobustScaler(), BayesianRidge())),
            ("lgb_fpall", "fp_all", lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.02, num_leaves=63, random_state=SEED, verbose=-1)),
            ("lgb_flagship", "flagship", lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.02, num_leaves=63, random_state=SEED, verbose=-1)),
            ("lgb_flagship_hbr", "flagship", lgb.LGBMRegressor(objective="huber", alpha=0.9, n_estimators=2000, learning_rate=0.02, num_leaves=63, random_state=SEED, verbose=-1)),
            ("lgb_flagship_quantile", "flagship", lgb.LGBMRegressor(objective="quantile", alpha=0.5, n_estimators=2000, learning_rate=0.02, num_leaves=63, random_state=SEED, verbose=-1)),
            ("lgb_research3d", "research3d", lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.02, num_leaves=63, random_state=SEED, verbose=-1)),
            ("xgb_master", "flagship", xgb.XGBRegressor(n_estimators=2000, learning_rate=0.02, max_depth=6, random_state=SEED, tree_method="hist")),
            ("cat_master", "flagship", CatBoostRegressor(iterations=2000, learning_rate=0.02, depth=6, verbose=False, random_seed=SEED)),
        ]
        
        oof_matrix = np.zeros((len(y_fit), len(models)))
        test_matrix = np.zeros((len(test_wide), len(models)))
        
        kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
        for m_idx, (m_name, f_set, model) in enumerate(models):
            X_tr, X_te = feature_sets[f_set][0][valid_idx], feature_sets[f_set][1]
            test_accum = np.zeros(len(X_te))
            for tr_i, va_i in kf.split(X_tr):
                xtr, ytr = X_tr[tr_i], y_fit[tr_i]
                xva, yva = X_tr[va_i], y_fit[va_i]
                if "lgb" in m_name:
                    model.fit(xtr, ytr, eval_set=[(xva, yva)], callbacks=[lgb.early_stopping(50, verbose=False)])
                elif "cat" in m_name:
                    model.fit(xtr, ytr, eval_set=(xva, yva), early_stopping_rounds=50)
                elif "xgb" in m_name:
                    model.fit(xtr, ytr, eval_set=[(xva, yva)], verbose=False)
                else:
                    model.fit(xtr, ytr)
                oof_matrix[va_i, m_idx] = model.predict(xva)
                test_accum += model.predict(X_te) / 5.0
            test_matrix[:, m_idx] = test_accum
            print(f"  {m_name:22s} OOF R2={r2_score(y_fit, oof_matrix[:, m_idx]):.5f}")
            
        # ── Greedy Blend Optimization ──
        print("  Optimizing Blend Weights...")
        best_w = optimize_blend_weights(oof_matrix, y_fit)
        blend_test = np.average(test_matrix, axis=1, weights=best_w)
        
        # ── 100% Full-Data Refit ──
        print("  Executing 100% Full-Data Refit...")
        refit_test = np.zeros_like(blend_test)
        for m_idx, (m_name, f_set, model) in enumerate(models):
            if best_w[m_idx] > 0.01:
                X_tr, X_te = feature_sets[f_set][0][valid_idx], feature_sets[f_set][1]
                model.fit(X_tr, y_fit)
                refit_test += model.predict(X_te) * best_w[m_idx]
                
        # ── Pseudo Labeling ──
        print("  Executing Double Pseudo-Label Pass...")
        conf_mask = refit_test > np.percentile(refit_test, 15) 
        X_ps = feature_sets["flagship"][1][conf_mask]
        y_ps = refit_test[conf_mask]
        
        X_aug = np.vstack([feature_sets["flagship"][0][valid_idx], X_ps])
        y_aug = np.concatenate([y_fit, y_ps])
        
        ps_model = lgb.LGBMRegressor(n_estimators=2500, learning_rate=0.015, num_leaves=40, random_state=SEED, verbose=-1)
        ps_model.fit(X_aug, y_aug)
        final_preds = (0.7 * refit_test) + (0.3 * ps_model.predict(feature_sets["flagship"][1]))
        
        final_wide_preds[t] = qt.inverse_transform(final_preds.reshape(-1, 1)).flatten() if use_rg else final_preds
    
    # ── Physics Clipping (OOM Safe Direct Assignment) ──
    print("\n--- Applying Maxwell & Koopmans Theoretical Boundaries ---")
    
    final_targets = []
    # Test_wide has the exact same row structure/index as test_long
    for idx, row in test_long.iterrows():
        tt = row['target_type'].lower()
        val = final_wide_preds.loc[idx, tt]
        
        if tt == 'nc': val = max(1.0, val) 
        if tt in ['egc', 'egb', 'eps']: val = max(0.0, val) 
        if tt == 'eps': val = max(val, final_wide_preds.loc[idx, 'nc']**2)
        if tt == 'eea': val = min(val, final_wide_preds.loc[idx, 'ei'])
        
        final_targets.append(val)
        
    pd.DataFrame({'id': test_long['id'], 'target': final_targets}).to_csv("submission.csv", index=False)
    print(f"\n✅ Titan Apex Pipeline Complete. Saved to submission.csv.")

if __name__ == "__main__":
    main()

System Initialized. Execution Device: cuda

--- Pivoting to Multi-Task Wide Format ---


Pivoting to Wide Format:   0%|          | 0/5920 [00:00<?, ?it/s]


--- Extracting All Physics & Topology ---


Physics & Graphs:   0%|          | 0/10860 [00:00<?, ?it/s]

Partitioning Independent Feature Matrices...

--- Training Deep SMILES NLP & GNN ---
Extracting CharCNN Embeddings safely...
Training MolGNN...
Extracting GNN Embeddings safely...

Training 12-Model Titan Ensemble for: EGC
  ridge_count            OOF R2=0.90760
  bayes_count            OOF R2=0.91512
  lgb_fpall              OOF R2=0.92248
  lgb_flagship           OOF R2=0.92332
  lgb_flagship_hbr       OOF R2=0.92446
  lgb_flagship_quantile  OOF R2=0.91671
  lgb_research3d         OOF R2=0.92316
  xgb_master             OOF R2=0.92308
  cat_master             OOF R2=0.92909
  Optimizing Blend Weights...
  Executing 100% Full-Data Refit...
  Executing Double Pseudo-Label Pass...

Training 12-Model Titan Ensemble for: EGB
  ridge_count            OOF R2=0.92732
  bayes_count            OOF R2=0.94089
  lgb_fpall              OOF R2=0.95010
  lgb_flagship           OOF R2=0.95015
  lgb_flagship_hbr       OOF R2=0.95100
  lgb_flagship_quantile  OOF R2=0.93803
  lgb_research3d         OOF